# EDA: Riot wiki reranker, Wikipedia Clickstream proxy

Train split only (features 2026-07, labels 2026-08). Test rows are never profiled. Built by `modeling/build_dataset.py`.

In [1]:
import polars as pl
from pathlib import Path
D = Path('../../../data/wikipedia-clickstream/modeling')
pairs = pl.read_parquet(D / 'pairs_train.parquet')
q = pl.read_parquet(D / 'queries_train.parquet')
print(pairs.shape, f'{pairs.estimated_size("mb"):.1f} MB')
pairs.head()

(5664045, 9) 156.9 MB


q,item,cand_opened_next,cand_twohop,cand_reverse,cand_popular,r,label,bucket
i32,i32,bool,bool,bool,bool,i64,f64,str
10,118200,false,false,false,true,0,0.0,"""torso"""
10,131817,false,false,false,true,0,0.0,"""torso"""
10,148027,false,false,false,true,0,0.0,"""torso"""
10,166788,false,false,false,true,0,0.0,"""torso"""
10,195798,false,false,false,true,0,0.0,"""torso"""


## Nulls

In [2]:
pairs.null_count()

q,item,cand_opened_next,cand_twohop,cand_reverse,cand_popular,r,label,bucket
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0


## Label: zero-inflated and heavy-tailed

In [3]:
pairs.select(pos_rate=(pl.col('r') > 0).mean(), label_mean=pl.col('label').mean())

pos_rate,label_mean
f64,f64
0.134167,0.497151


In [4]:
pairs.filter(pl.col('r') > 0).select([pl.col('r').quantile(p).alias(f'r_p{int(p*1000)}') for p in (0.5, 0.9, 0.99, 0.999)] + [pl.col('r').max().alias('r_max')])

r_p500,r_p900,r_p990,r_p999,r_max
f64,f64,f64,f64,i64
30.0,190.0,1493.0,7559.0,140110


In [5]:
pairs.group_by('bucket').agg(pairs=pl.len(), pos_rate=(pl.col('r') > 0).mean(), label_mean=pl.col('label').mean()).sort('bucket')

bucket,pairs,pos_rate,label_mean
str,u32,f64,f64
"""cold""",248826,0.014363,0.03896
"""head""",1205146,0.327094,1.343657
"""tail""",1087850,0.040856,0.118676
"""torso""",3122223,0.101758,0.338792


## Queries and candidates per query

In [6]:
q.group_by('bucket').agg(n=pl.len(), seen_f=pl.col('seen_f').mean(), src_out_f_median=pl.col('src_out_f').median()).sort('bucket')

bucket,n,seen_f,src_out_f_median
str,u32,f64,f64
"""cold""",11615,0.696427,0.0
"""head""",12150,1.0,2429.5
"""tail""",30072,1.0,23.0
"""torso""",46163,1.0,168.0


In [7]:
pairs.group_by('q').agg(n_cand=pl.len(), n_pos=(pl.col('r') > 0).sum(), bucket=pl.col('bucket').first()).group_by('bucket').agg(cand_median=pl.col('n_cand').median(), pos_median=pl.col('n_pos').median(), no_pos_share=(pl.col('n_pos') == 0).mean()).sort('bucket')

bucket,cand_median,pos_median,no_pos_share
str,f64,f64,f64
"""cold""",21.0,0.0,0.719156
"""head""",99.0,30.0,0.0
"""tail""",33.0,1.0,0.04007
"""torso""",73.0,5.0,0.000563


## Candidate sources

In [8]:
flags = [c for c in pairs.columns if c.startswith('cand_')]
pl.DataFrame([{ 'source': f, 'share_of_pairs': pairs[f].mean(), 'pos_rate': pairs.filter(pl.col(f))['r'].gt(0).mean()} for f in flags])

source,share_of_pairs,pos_rate
str,f64,f64
"""cand_opened_next""",0.144855,0.854302
"""cand_twohop""",0.49581,0.092979
"""cand_reverse""",0.110236,0.445376
"""cand_popular""",0.353105,0.000222


## Candidate recall (from build stats)

In [9]:
import json
json.load(open('../modeling/build_dataset_stats.json'))['train']['candidate_recall']

{'cold': {'recall': 0.1829, 'recall_w': 0.0624},
 'torso': {'recall': 0.8512, 'recall_w': 0.9327},
 'head': {'recall': 0.6713, 'recall_w': 0.9185},
 'tail': {'recall': 0.7601, 'recall_w': 0.7982}}